In [1]:
"""
Cash-flow matching y análisis de sensibilidad
Resuelve el mismo modelo que el archivo de Excel (Solver, Simplex LP).

Requisitos:  pip install numpy scipy
Opcional (rangos permisibles como en el informe de Solver):  pip install highspy

Cantidades en miles de unidades monetarias.
"""

import numpy as np
from scipy.optimize import linprog

# ----------------------------------------------------------------------
# 1. Datos del problema
# ----------------------------------------------------------------------
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo"]
N_BASE = [110, 90, -180, 160, -260]   # + necesidad, - superávit

R_LINEA = 0.01     # línea de crédito, 1% mensual, se paga al mes siguiente
R_PAPEL = 0.022    # papel comercial, 2.2% total, vence 3 meses después
R_INV = 0.004      # inversión de excedentes, 0.4% a un mes
LIM_LINEA = 80     # máximo mensual de la línea de crédito

# Orden de las variables de decisión
VARS = ["x1", "x2", "x3", "x4",   # línea de crédito ene-abr (en mayo no hay deuda nueva)
        "y1", "y2",               # papel comercial ene y feb
        "z1", "z2", "z3", "z4",   # inversión de excedentes ene-abr
        "v"]                      # riqueza final de mayo
DESC = {
    "x1": "Línea de crédito enero", "x2": "Línea de crédito febrero",
    "x3": "Línea de crédito marzo", "x4": "Línea de crédito abril",
    "y1": "Papel comercial enero", "y2": "Papel comercial febrero",
    "z1": "Inversión enero", "z2": "Inversión febrero",
    "z3": "Inversión marzo", "z4": "Inversión abril",
    "v": "Riqueza final mayo",
}
idx = {n: i for i, n in enumerate(VARS)}


def construir_modelo():
    """Matriz de balances: fuentes - usos = necesidad del mes."""
    a, b, c = 1 + R_LINEA, 1 + R_PAPEL, 1 + R_INV
    filas = [
        {"x1": 1, "y1": 1, "z1": -1},                                   # Enero
        {"x2": 1, "y2": 1, "x1": -a, "z1": c, "z2": -1},                # Febrero
        {"x3": 1, "x2": -a, "z2": c, "z3": -1},                         # Marzo
        {"x4": 1, "x3": -a, "y1": -b, "z3": c, "z4": -1},               # Abril
        {"x4": -a, "y2": -b, "z4": c, "v": -1},                         # Mayo
    ]
    A = np.zeros((len(filas), len(VARS)))
    for i, fila in enumerate(filas):
        for var, coef in fila.items():
            A[i, idx[var]] = coef
    # Cotas: 0 <= x_t <= 80, el resto >= 0
    cotas = [(0, LIM_LINEA) if n.startswith("x") else (0, None) for n in VARS]
    # linprog minimiza, así que max v  ==  min -v
    costo = np.zeros(len(VARS))
    costo[idx["v"]] = -1
    return costo, A, cotas


def resolver(N):
    costo, A, cotas = construir_modelo()
    res = linprog(costo, A_eq=A, b_eq=N, bounds=cotas, method="highs")
    if not res.success:
        raise RuntimeError(res.message)
    riqueza = -res.fun
    # Convertimos los signos al formato de Solver (problema de maximización)
    precios_sombra = -res.eqlin.marginals
    costos_reducidos = -(res.lower.marginals + res.upper.marginals)
    costos_reducidos[np.abs(costos_reducidos) < 1e-12] = 0.0   # evita "-0.000000"
    return riqueza, res.x, precios_sombra, costos_reducidos


def rangos_permisibles(N):
    """Aumento/disminución permisible del lado derecho (como Solver).
    Usa HiGHS directamente; si no está instalado, regresa None."""
    try:
        import highspy
    except ImportError:
        return None
    costo, A, cotas = construir_modelo()
    h = highspy.Highs()
    h.setOptionValue("output_flag", False)
    inf = highspy.kHighsInf
    for (lo, hi), cj in zip(cotas, costo):
        h.addVar(lo, inf if hi is None else hi)
    h.changeColsCost(len(VARS), np.arange(len(VARS)), costo)
    for i, bi in enumerate(N):
        cols = np.nonzero(A[i])[0]
        h.addRow(bi, bi, len(cols), cols, A[i, cols])
    h.run()
    _, rg = h.getRanging()
    aum = [rg.row_bound_up.value_[i] - N[i] for i in range(len(N))]
    dis = [N[i] - rg.row_bound_dn.value_[i] for i in range(len(N))]
    return aum, dis


def fmt_rango(x):
    return "1E+30" if np.isinf(x) else f"{x:10.4f}"


# ----------------------------------------------------------------------
# 2. Solución óptima
# ----------------------------------------------------------------------
v_base, x_opt, ps, cr = resolver(N_BASE)

print("=" * 66)
print("SOLUCIÓN ÓPTIMA")
print("=" * 66)
for n, val in zip(VARS, x_opt):
    print(f"  {n:>3}  {DESC[n]:<26} {val:10.4f}")
print(f"\n  Riqueza final máxima v* = {v_base:.4f}")

# ----------------------------------------------------------------------
# 3. Informe de sensibilidad
# ----------------------------------------------------------------------
print("\n" + "=" * 66)
print("VARIABLES: costos reducidos")
print("=" * 66)
print(f"  {'Var':>3}  {'Valor':>10}  {'Costo reducido':>15}")
for n, val, r in zip(VARS, x_opt, cr):
    print(f"  {n:>3}  {val:10.4f}  {r:15.6f}")

rangos = rangos_permisibles(N_BASE)
print("\n" + "=" * 66)
print("RESTRICCIONES: precios sombra")
print("=" * 66)
print(f"  {'Balance':<9} {'RHS':>6} {'Precio sombra':>14} {'Aum. perm.':>11} {'Dism. perm.':>11}")
for i, m in enumerate(MESES):
    if rangos:
        a, d = fmt_rango(rangos[0][i]), fmt_rango(rangos[1][i])
    else:
        a = d = "n/d"
    print(f"  {m:<9} {N_BASE[i]:>6} {ps[i]:14.5f} {a:>11} {d:>11}")
if not rangos:
    print("  (Instala highspy para ver los rangos permisibles: pip install highspy)")


# ----------------------------------------------------------------------
# 4. Escenarios A y B
# ----------------------------------------------------------------------
def escenario(nombre, mes, nuevo_rhs):
    i = MESES.index(mes)
    delta = nuevo_rhs - N_BASE[i]
    print("\n" + "=" * 66)
    print(f"ESCENARIO {nombre}: {mes} {N_BASE[i]} -> {nuevo_rhs}  (Δ = {delta:+})")
    print("=" * 66)
    if rangos:
        dentro = -rangos[1][i] <= delta <= rangos[0][i]
        print(f"  Rango permisible: -{rangos[1][i]:.4f} a +{rangos[0][i]:.4f}"
              f"  ->  {'dentro' if dentro else 'FUERA'} del rango")
    estimada = v_base + ps[i] * delta
    N_nuevo = list(N_BASE)
    N_nuevo[i] = nuevo_rhs
    real, x_nuevo, _, _ = resolver(N_nuevo)
    print(f"  Estimación con precio sombra: {v_base:.4f} + ({ps[i]:.5f})({delta}) = {estimada:.4f}")
    print(f"  Riqueza re-optimizada:        {real:.4f}")
    print(f"  Diferencia:                   {estimada - real:.6f}")
    print("  Nuevo plan:", ", ".join(f"{n}={x:.2f}" for n, x in zip(VARS, x_nuevo) if x > 1e-9))


escenario("A", "Enero", 120)
escenario("B", "Marzo", -160)

# ----------------------------------------------------------------------
# 5. Costo reducido de x2 (interpretación)
# ----------------------------------------------------------------------
print("\n" + "=" * 66)
print("COSTO REDUCIDO DE x2 (línea de crédito en febrero)")
print("=" * 66)
costo_linea = (1 + R_LINEA) * -ps[MESES.index("Marzo")]
costo_papel = 1 + R_PAPEL
print(f"  Costo de financiar febrero con línea (1.01 x valor de marzo): {costo_linea:.5f}")
print(f"  Costo de financiar febrero con papel comercial:              {costo_papel:.5f}")
print(f"  Diferencia (= |costo reducido|):                              {costo_linea - costo_papel:.5f}")
print(f"  x2 entraría si la tasa de la línea bajara a menos de "
      f"{(costo_papel / -ps[MESES.index('Marzo')] - 1) * 100:.3f}% mensual")

SOLUCIÓN ÓPTIMA
   x1  Línea de crédito enero        11.4481
   x2  Línea de crédito febrero       0.0000
   x3  Línea de crédito marzo         0.0000
   x4  Línea de crédito abril        80.0000
   y1  Papel comercial enero         98.5519
   y2  Papel comercial febrero      101.5626
   z1  Inversión enero                0.0000
   z2  Inversión febrero              0.0000
   z3  Inversión marzo              180.0000
   z4  Inversión abril                0.0000
    v  Riqueza final mayo            75.4030

  Riqueza final máxima v* = 75.4030

VARIABLES: costos reducidos
  Var       Valor   Costo reducido
   x1     11.4481         0.000000
   x2      0.0000        -0.002180
   x3      0.0000        -0.006060
   x4     80.0000         0.000000
   y1     98.5519         0.000000
   y2    101.5626         0.000000
   z1      0.0000        -0.006132
   z2      0.0000        -0.003904
   z3    180.0000         0.000000
   z4      0.0000        -0.006000
    v     75.4030         0.000000

RE